Menyiapkan Dataset
-

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


Mengimpor Library
-

In [2]:
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("Seluruh library berhasil diimpor.")

Seluruh library berhasil diimpor.


Membuat SparkSession
-

In [3]:
spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Mengatur jumlah partisi shuffle
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat.")
print("Versi Spark:", spark.version)

26/11/12 14:05:00 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 192.168.50.46 instead (on interface wlo1)
26/11/12 14:05:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:05:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat.
Versi Spark: 3.5.9


Membaca Dataset CSV
-

In [4]:
df_tugas = spark.read.csv(
    "tugas11_penjualan.csv",
    header=True,
    inferSchema=True
)

# Menampilkan 5 data pertama
df_tugas.show(5)

# Menampilkan struktur kolom
df_tugas.printSchema()

# Menghitung jumlah data
print("Total baris:", df_tugas.count())

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

Total baris: 2500


A. Feature Engineering
-

1. Membuat StringIndexer

In [5]:
# Membuat StringIndexer untuk nama_menu
indexer_menu = StringIndexer(
    inputCol="nama_menu",
    outputCol="nama_menu_idx"
)

# Membuat StringIndexer untuk cabang
indexer_cabang = StringIndexer(
    inputCol="cabang",
    outputCol="cabang_idx"
)

# Membuat StringIndexer untuk waktu_pesan
indexer_waktu = StringIndexer(
    inputCol="waktu_pesan",
    outputCol="waktu_pesan_idx"
)

# Melakukan fit dan transform untuk setiap kolom
indexer_menu_model = indexer_menu.fit(df_tugas)
df_indexed_tugas = indexer_menu_model.transform(df_tugas)

indexer_cabang_model = indexer_cabang.fit(df_indexed_tugas)
df_indexed_tugas = indexer_cabang_model.transform(df_indexed_tugas)

indexer_waktu_model = indexer_waktu.fit(df_indexed_tugas)
df_indexed_tugas = indexer_waktu_model.transform(df_indexed_tugas)

# Menampilkan hasil indexing
df_indexed_tugas.select(
    "nama_menu",
    "nama_menu_idx",
    "cabang",
    "cabang_idx",
    "waktu_pesan",
    "waktu_pesan_idx"
).show(10)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
|     Kopi Susu|          1.0|Yogyakarta|       1.0|       Pagi|            0.0|
|     Americano|          0.

2. Menggabungkan Seluruh Fitur

In [6]:
# Membuat VectorAssembler untuk menggabungkan seluruh fitur
assembler_utama_tugas = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan",
        "nama_menu_idx",
        "cabang_idx",
        "waktu_pesan_idx"
    ],
    outputCol="features"
)

# Membuat kolom features
df_tugas_final = assembler_utama_tugas.transform(df_indexed_tugas)

# Menampilkan hasil feature engineering
df_tugas_final.select(
    "features",
    "total_penjualan"
).show(10, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
|(5,[0,1],[1.0,30000.0])  |32462.966390290698|
|[2.0,25000.0,2.0,0.0,1.0]|51846.84285104993 |
|[5.0,25000.0,2.0,2.0,0.0]|114192.26376783242|
|[1.0,22000.0,1.0,1.0,0.0]|9697.180344260187 |
|(5,[0,1],[2.0,15000.0])  |32316.051747436813|
+-------------------------+------------------+
only showing top 10 rows



B. Train/Test Split & Pelatihan Model
-

1. Membagi Dataset

In [7]:
# Membagi dataset menjadi 75% training dan 25% testing
train_tugas, test_tugas = df_tugas_final.randomSplit(
    [0.75, 0.25],
    seed=101
)

# Menampilkan jumlah data training dan testing
print("Jumlah data training:", train_tugas.count())
print("Jumlah data testing :", test_tugas.count())

Jumlah data training: 1839
Jumlah data testing : 661


2. Melatih Model Regresi Linear

In [8]:
# Mendefinisikan model LinearRegression
lr_tugas = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan"
)

# Melatih model menggunakan data training
model_tugas = lr_tugas.fit(train_tugas)

print("Model regresi linear berhasil dilatih.")

# Menampilkan koefisien dan intercept
print("Koefisien:", model_tugas.coefficients)
print("Intercept :", model_tugas.intercept)

Model regresi linear berhasil dilatih.
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept : -66903.17766485889


C. Evaluasi Model
-

1. Membuat Prediksi

In [9]:
# Membuat prediksi menggunakan data testing
prediksi_tugas = model_tugas.transform(test_tugas)

# Menampilkan hasil prediksi
prediksi_tugas.select(
    "nama_menu",
    "cabang",
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10)

+---------+--------+---+------------+------------------+------------------+
|nama_menu|  cabang|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+---+------------+------------------+------------------+
|Americano|Magelang|  1|       25000|19721.170446051627|31056.146105506836|
|Americano|Magelang|  1|       25000|  21236.0956379149|31056.146105506836|
|Americano|Magelang|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|  5|       22000|115810.29600214415|110174.03658512786|
|Americano|Magelang|  5|       25000|140386.40273615732|119242.79918960987|
|Americano|Magelang|  2|       30000| 73842.64420790318| 67877.77492201277|
|Americano|Magelang|  3|       15000|32345.642983082646| 44580.62517062847|
|Americano|Magelang|  4|       15000| 45510.16712269087| 66627.28844165422|
+---------+-

2. Menghitung RMSE dan R²

In [10]:
# Membuat evaluator untuk menghitung RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

# Menghitung nilai RMSE
rmse_tugas = evaluator_rmse.evaluate(prediksi_tugas)

# Membuat evaluator untuk menghitung R2
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

# Menghitung nilai R2
r2_tugas = evaluator_r2.evaluate(prediksi_tugas)

# Menampilkan hasil evaluasi
print(f"RMSE: Rp {rmse_tugas:,.2f}")
print(f"R2  : {r2_tugas:.4f}")
print(f"Persentase variasi yang dijelaskan: {r2_tugas * 100:.2f}%")

RMSE: Rp 10,900.80
R2  : 0.9107
Persentase variasi yang dijelaskan: 91.07%


Berdasarkan hasil, nilai RMSE menunjukkan besarnya kesalahan prediksi total penjualan dalam satuan rupiah. Sementara itu, nilai R² menunjukkan seberapa besar variasi total penjualan yang dapat dijelaskan oleh model. Model dapat dikatakan memiliki performa yang baik apabila nilai R² mendekati 1 dan nilai RMSE relatif kecil dibandingkan dengan nilai total penjualan.

D. Eksperimen Perbandingan Fitur
-

1. Membuat VectorAssembler Sederhana

In [11]:
# Membuat VectorAssembler dengan dua fitur saja
assembler_sederhana_tugas = VectorAssembler(
    inputCols=[
        "qty",
        "harga_satuan"
    ],
    outputCol="features_sederhana"
)

# Mengubah data training menggunakan dua fitur
train_tugas_sederhana = assembler_sederhana_tugas.transform(train_tugas)

# Mengubah data testing menggunakan dua fitur
test_tugas_sederhana = assembler_sederhana_tugas.transform(test_tugas)

# Menampilkan hasil penggabungan fitur
train_tugas_sederhana.select(
    "qty",
    "harga_satuan",
    "features_sederhana"
).show(5, truncate=False)

+---+------------+------------------+
|qty|harga_satuan|features_sederhana|
+---+------------+------------------+
|1  |15000       |[1.0,15000.0]     |
|1  |20000       |[1.0,20000.0]     |
|1  |22000       |[1.0,22000.0]     |
|1  |25000       |[1.0,25000.0]     |
|1  |25000       |[1.0,25000.0]     |
+---+------------+------------------+
only showing top 5 rows



2. Melatih Model Kedua

In [12]:
# Mendefinisikan model regresi linear sederhana
lr_sederhana_tugas = LinearRegression(
    featuresCol="features_sederhana",
    labelCol="total_penjualan"
)

# Melatih model menggunakan dua fitur
model_sederhana_tugas = lr_sederhana_tugas.fit(
    train_tugas_sederhana
)

print("Model regresi sederhana berhasil dilatih.")

# Membuat prediksi menggunakan data testing
prediksi_sederhana_tugas = model_sederhana_tugas.transform(
    test_tugas_sederhana
)

# Menampilkan hasil prediksi
prediksi_sederhana_tugas.select(
    "qty",
    "harga_satuan",
    "total_penjualan",
    "prediction"
).show(10)

Model regresi sederhana berhasil dilatih.
+---+------------+------------------+------------------+
|qty|harga_satuan|   total_penjualan|        prediction|
+---+------------+------------------+------------------+
|  1|       25000|19721.170446051627| 31177.71885185805|
|  1|       25000|  21236.0956379149| 31177.71885185805|
|  4|       22000|  85834.8564401191| 88260.08378640364|
|  4|       22000|102174.09100317833| 88260.08378640364|
|  4|       30000|113044.63431555482|112428.48226630976|
|  5|       22000|115810.29600214415|110308.58857457378|
|  5|       25000|140386.40273615732|119371.73800453858|
|  2|       30000| 73842.64420790318| 68331.47268996952|
|  3|       15000|32345.642983082646| 45064.23032831565|
|  4|       15000| 45510.16712269087| 67112.73511648577|
+---+------------+------------------+------------------+
only showing top 10 rows



3. Menghitung dan Membandingkan Nilai R²

In [13]:
# Menghitung R2 model sederhana
evaluator_r2_sederhana_tugas = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)

r2_sederhana_tugas = evaluator_r2_sederhana_tugas.evaluate(
    prediksi_sederhana_tugas
)

# Menampilkan perbandingan R2 kedua model
print("R2 Model Utama     :", round(r2_tugas, 4))
print("R2 Model Sederhana :", round(r2_sederhana_tugas, 4))

# Membandingkan hasil kedua model
selisih_r2 = r2_tugas - r2_sederhana_tugas

print("Selisih R2:", round(selisih_r2, 4))

if selisih_r2 > 0:
    print("Model utama memiliki nilai R2 lebih tinggi.")
elif selisih_r2 < 0:
    print("Model sederhana memiliki nilai R2 lebih tinggi.")
else:
    print("Kedua model memiliki nilai R2 yang sama.")

R2 Model Utama     : 0.9107
R2 Model Sederhana : 0.9108
Selisih R2: -0.0001
Model sederhana memiliki nilai R2 lebih tinggi.


Berdasarkan hasil perbandingan, model utama menggunakan 5 fitur, sedangkan model sederhana hanya menggunakan 2 fitur, yaitu qty dan harga_satuan. Penambahan informasi menu, cabang, dan waktu pemesanan perlu dilihat berdasarkan perubahan nilai R². Apabila nilai R² kedua model hampir sama, berarti penambahan fitur tersebut belum memberikan peningkatan akurasi prediksi yang berarti.

E. Prediksi Skenario Baru
-

1. Membuat DataFrame Skenario Baru

In [14]:
# Membuat tiga skenario pesanan baru
data_skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Cappuccino", "Yogyakarta", "Siang", 3, 25000),
    ("Roti Bakar", "Semarang", "Malam", 1, 30000)
]

# Mengubah data skenario menjadi Spark DataFrame
df_skenario = spark.createDataFrame(
    data_skenario,
    [
        "nama_menu",
        "cabang",
        "waktu_pesan",
        "qty",
        "harga_satuan"
    ]
)

# Menampilkan data skenario
df_skenario.show()

+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Magelang|       Pagi|  2|       18000|
|Cappuccino|Yogyakarta|      Siang|  3|       25000|
|Roti Bakar|  Semarang|      Malam|  1|       30000|
+----------+----------+-----------+---+------------+



2. StringIndexer pada Data Skenario

In [15]:
# Menggunakan model StringIndexer yang sudah di-fit sebelumnya
df_skenario_indexed = indexer_menu_model.transform(df_skenario)

df_skenario_indexed = indexer_cabang_model.transform(
    df_skenario_indexed
)

df_skenario_indexed = indexer_waktu_model.transform(
    df_skenario_indexed
)

# Menampilkan hasil indexing
df_skenario_indexed.select(
    "nama_menu",
    "nama_menu_idx",
    "cabang",
    "cabang_idx",
    "waktu_pesan",
    "waktu_pesan_idx"
).show()

+----------+-------------+----------+----------+-----------+---------------+
| nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+----------+-------------+----------+----------+-----------+---------------+
| Kopi Susu|          1.0|  Magelang|       0.0|       Pagi|            0.0|
|Cappuccino|          2.0|Yogyakarta|       1.0|      Siang|            2.0|
|Roti Bakar|          4.0|  Semarang|       2.0|      Malam|            1.0|
+----------+-------------+----------+----------+-----------+---------------+



3. Menggabungkan Fitur Menggunakan VectorAssembler

In [16]:
# Menggunakan VectorAssembler yang sama seperti data training
df_skenario_final = assembler_utama_tugas.transform(
    df_skenario_indexed
)

# Menampilkan hasil penggabungan fitur
df_skenario_final.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "features"
).show(truncate=False)

+----------+----------+-----------+-------------------------+
|nama_menu |cabang    |waktu_pesan|features                 |
+----------+----------+-----------+-------------------------+
|Kopi Susu |Magelang  |Pagi       |[2.0,18000.0,1.0,0.0,0.0]|
|Cappuccino|Yogyakarta|Siang      |[3.0,25000.0,2.0,1.0,2.0]|
|Roti Bakar|Semarang  |Malam      |[1.0,30000.0,4.0,2.0,1.0]|
+----------+----------+-----------+-------------------------+



4. Prediksi Total Penjualan

In [17]:
# Melakukan prediksi menggunakan model utama
hasil_prediksi_skenario = model_tugas.transform(
    df_skenario_final
)

# Menampilkan hasil prediksi ketiga skenario
hasil_prediksi_skenario.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098144|
|Cappuccino|Yogyakarta|Siang      |3  |25000       |75521.96745552517 |
|Roti Bakar|Semarang  |Malam      |1  |30000       |46236.46247159745 |
+----------+----------+-----------+---+------------+------------------+



Menutup SparkSession
-

In [18]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
